# Topological Event Study by Event Category

Supplementary Experiment 2: analyse the *trajectory* of the best TDA indicator
(log-DTW Betti-0) around events grouped by category.

Reveals mechanism differences: rising topology (network fragmentation) vs
falling topology (network consolidation) preceding different shock types.

In [ ]:
from pathlib import Path
from datetime import timedelta
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('/Users/jane/Documents/202511吾-Systems/3.Data/TDA_SP500_VIX_BTC_BVOL_CVI_merged0329_labeled_with_network.csv')
DATA_OUT  = Path('/Users/jane/Documents/202511吾-Systems/12.Data0329')

EVENT_CATALOG = [
    ('2020-12-01','Protocol/Tech','Ethereum 2.0'),
    ('2021-01-02','Market Milestone','BTC $30k'),('2021-01-07','Market Milestone','BTC $40k'),
    ('2021-01-29','Market Milestone','Dogecoin frenzy'),('2021-02-16','Market Milestone','BTC $50k'),
    ('2021-03-13','Market Milestone','BTC $60k'),('2021-04-10','Market Milestone','BTC peak'),
    ('2021-05-12','Corporate Action','Musk BTC suspend'),('2021-05-17','Corporate Action','Musk tweet'),
    ('2021-05-18','Corporate Action','Musk clarifies'),('2021-05-19','Regulation','China crackdown'),
    ('2021-06-09','Regulation','El Salvador'),('2021-09-24','Regulation','China ban'),
    ('2021-10-15','Institutional Adoption','US BTC ETF'),('2021-11-15','Regulation','US Infra Bill'),
    ('2022-04-27','Regulation','CAR BTC'),('2022-05-01','Systemic Failure','UST depeg'),
    ('2022-05-11','Systemic Failure','UST collapse'),('2022-05-12','Systemic Failure','Terra collapse'),
    ('2022-05-13','Systemic Failure','Luna delisting'),('2022-07-20','Corporate Action','Tesla sells BTC'),
    ('2022-11-01','Systemic Failure','FTX collapse'),('2023-03-01','Systemic Failure','SVB stress'),
    ('2023-03-10','Systemic Failure','USDC depeg'),('2023-05-17','Corporate Action','Tether buys BTC'),
    ('2023-06-16','Institutional Adoption','BlackRock ETF'),('2023-07-01','Security/Hack','DeFi hacks'),
    ('2023-10-01','Regulation','Grayscale'),('2024-03-19','Institutional Adoption','Japan GPIF'),
    ('2024-04-20','Protocol/Tech','BTC halving'),('2025-01-20','Regulation','Trump inaug.'),
    ('2025-02-03','Macro Shock','AI/tariff shock'),('2025-02-21','Security/Hack','Bybit hack'),
    ('2025-03-07','Regulation','BTC Reserve'),('2025-05-20','Regulation','Stablecoin Bill'),
    ('2025-06-05','Institutional Adoption','Circle IPO'),('2025-06-17','Regulation','GENIUS Act'),
]
events_df = pd.DataFrame(EVENT_CATALOG, columns=['date','category','event'])
events_df['date'] = pd.to_datetime(events_df['date'])
CAT_COLORS = {
    'Regulation':'#E64B35','Market Milestone':'#4DBBD5','Systemic Failure':'#F39B7F',
    'Corporate Action':'#91D1C2','Institutional Adoption':'#3C5488',
    'Protocol/Tech':'#00A087','Macro Shock':'#8491B4','Security/Hack':'#B09C85',
}
CATS_ORDERED = ['Regulation','Systemic Failure','Market Milestone','Institutional Adoption',
                'Corporate Action','Protocol/Tech','Macro Shock','Security/Hack']
FEATURE  = 'log_dtw_betti0'
PRE_DAYS = 30; POST_DAYS = 10
DAYS = np.arange(-PRE_DAYS, POST_DAYS+1)


In [ ]:
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
feat_z = (df[FEATURE] - df[FEATURE].mean()) / df[FEATURE].std()

traj_rows = []
for _, ev in events_df.iterrows():
    ev_idx = df.index[df['date']==ev['date']]
    ev_i = ev_idx[0] if len(ev_idx)>0 else (df['date']-ev['date']).abs().idxmin()
    for d in DAYS:
        idx = ev_i + d
        if 0 <= idx < len(df):
            traj_rows.append({'category':ev['category'],'event':ev['event'],
                               'day':d,'z':feat_z.iloc[idx]})

traj_df = pd.DataFrame(traj_rows)
traj_df.to_csv(DATA_OUT/'tbl_tda_event_study_trajectories.csv', index=False)
avg_traj = traj_df.groupby(['category','day'])['z'].agg(['mean','sem']).reset_index()

pre_mean  = traj_df[traj_df['day'].between(-21,-1)].groupby('category')['z'].mean().round(3)
post_mean = traj_df[traj_df['day'].between(0,7)].groupby('category')['z'].mean().round(3)
pre_std   = traj_df[traj_df['day'].between(-21,-1)].groupby('category')['z'].std().round(3)
summary   = pd.DataFrame({'mean_z_pre[-21,-1]':pre_mean,'std_z_pre[-21,-1]':pre_std,
                           'mean_z_post[0,+7]':post_mean}).reset_index()
summary['signal_direction'] = np.where(summary['mean_z_pre[-21,-1]']>0,'Rising (↑)','Falling (↓)')
summary.to_csv(DATA_OUT/'tbl_tda_event_study_summary.csv', index=False)
display(summary)


In [ ]:
cats_plot = [c for c in CATS_ORDERED if events_df[events_df['category']==c].shape[0]>=2]
n_cols = 4
n_rows = int(np.ceil(len(cats_plot)/n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4.5*n_rows), sharey=False, sharex=True)
axes = axes.flatten()
for ax, cat in zip(axes, cats_plot):
    sub = avg_traj[avg_traj['category']==cat].sort_values('day')
    n_ev = events_df[events_df['category']==cat].shape[0]
    color = CAT_COLORS[cat]
    ax.axvspan(-21,0,color='#eaf4fb',alpha=0.5,zorder=0)
    ax.axhline(0,color='grey',linewidth=0.8,linestyle='--',alpha=0.6)
    ax.axvline(0,color='red',linewidth=1.2,linestyle='-',alpha=0.7)
    ax.fill_between(sub['day'],sub['mean']-sub['sem'],sub['mean']+sub['sem'],alpha=0.25,color=color)
    ax.plot(sub['day'],sub['mean'],color=color,linewidth=2.2,zorder=5)
    ax.set_title(f'{cat}  (n={n_ev})',fontsize=10,fontweight='bold',color=color,pad=4)
    ax.set_xlim(-PRE_DAYS,POST_DAYS)
    ax.xaxis.grid(True,linestyle='--',alpha=0.3); ax.yaxis.grid(True,linestyle='--',alpha=0.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for ax in axes[len(cats_plot):]: ax.set_visible(False)
fig.suptitle('Topological Event Study: log-DTW Betti-0 Trajectories by Event Category\n(mean ± 1 s.e.m., z-scored)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(DATA_OUT/'fig_tda_event_study_by_category.png', dpi=300, bbox_inches='tight')
fig.savefig(DATA_OUT/'fig_tda_event_study_by_category.pdf', bbox_inches='tight')
plt.show(); print('Saved: subplots figure.')


In [ ]:
fig3, ax3 = plt.subplots(figsize=(11, 5))
ax3.axvspan(-21,0,color='#eaf4fb',alpha=0.45,zorder=0)
ax3.axhline(0,color='grey',linewidth=0.8,linestyle='--',alpha=0.5)
ax3.axvline(0,color='red',linewidth=1.5,linestyle='-',alpha=0.7,label='Event day (day 0)')
for cat in cats_plot:
    sub = avg_traj[avg_traj['category']==cat].sort_values('day')
    n_ev = events_df[events_df['category']==cat].shape[0]
    ax3.plot(sub['day'],sub['mean'],color=CAT_COLORS[cat],linewidth=2.0,label=f'{cat} (n={n_ev})',zorder=5)
ax3.set_xlabel('Days relative to event date', fontsize=11)
ax3.set_ylabel('log-DTW Betti-0 (z-score)', fontsize=11)
ax3.set_xlim(-PRE_DAYS,POST_DAYS)
ax3.xaxis.grid(True,linestyle='--',alpha=0.3); ax3.yaxis.grid(True,linestyle='--',alpha=0.3)
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
ax3.legend(frameon=False, fontsize=9, loc='upper left', ncol=2)
ax3.set_title('Comparison of Mean Topological Trajectories Across Event Categories', fontsize=12.5, fontweight='bold')
fig3.tight_layout()
fig3.savefig(DATA_OUT/'fig_tda_event_study_overlay.png', dpi=300, bbox_inches='tight')
fig3.savefig(DATA_OUT/'fig_tda_event_study_overlay.pdf', bbox_inches='tight')
plt.show(); print('Saved: overlay figure.')
